# 💼 NFL Transactions - Bronze Layer Ingestion

## 🎯 Purpose

Ingest official NFL transactions (signings, IR moves, activations, releases) into `main.fantasai.bronze_nfl_transactions`.

---

## 🔗 Data Source

**Source:** ESPN Transactions API  
**Endpoint:** `https://site.api.espn.com/apis/site/v2/sports/football/nfl/transactions`  
**Cost:** ✅ FREE (no API key required)  
**Rate Limits:** ✅ None observed  
**Coverage:** Official NFL transactions from all 32 teams  

---

## 📊 Transaction Types

### High Fantasy Value:
* **🏅 Signed** - New player signings (waiver opportunities)
* **🚫 Waived/Released** - Players available on waivers
* **🏯 Traded** - Opportunity changes
* **🏥 IR - Reserve/Injured** - Long-term injuries
* **✅ Activated** - Players returning from IR/suspension
* **🔼 Practice Squad** - Potential call-ups

### Medium Fantasy Value:
* **🚨 Suspended** - Availability changes
* **📄 Contract Extension** - Long-term security
* **⏸️ Reserve List** - Various statuses

---

## 📋 Data Fields

* `transaction_id` - Unique transaction ID
* `transaction_date` - Date of transaction
* `transaction_type` - Type (Signed, Waived, IR, etc.)
* `player_name` - Player name
* `position` - Player position
* `team` - Team (from or to)
* `description` - Full transaction description
* `espn_player_id` - ESPN player ID (if available)
* `fetched_at` - Ingestion timestamp

---

## ⚙️ Execution

**Schedule:** Daily at 8:00 AM UTC  
**Runtime:** ~30 seconds  
**Lookback:** Last 30 days of transactions  
**Deduplication:** Skips existing transactions  

---

## 💡 Fantasy Use Cases

1. **Waiver Wire Alerts** - New signings = pickup opportunities
2. **Injury Tracking** - IR moves = season-ending injuries
3. **Depth Chart Changes** - Releases = more opportunities for teammates
4. **Trade Impact** - New team = new offensive system
5. **Practice Squad Promotions** - Emergency call-ups before game day

---

## 📝 Notes

* Transactions posted within hours of official announcements
* Some transactions lack player IDs (undrafted, practice squad)
* Multiple transaction types can occur same day for one player
* Historical data available via date range parameters

In [0]:
# =============================================================================
# NFL TRANSACTIONS INGESTION - CONFIGURATION
# =============================================================================

from datetime import datetime, timedelta
import time

print("="*80)
print("💼 NFL Transactions Ingestion Configuration")
print("="*80)

# === API CONFIGURATION ===
ESPN_TRANSACTIONS_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/transactions"
REQUEST_TIMEOUT = 15  # seconds

# === TABLE CONFIGURATION ===
BRONZE_TABLE = "main.fantasai.bronze_nfl_transactions"

# === LOOKBACK PERIOD ===
# Fetch transactions from last N days
LOOKBACK_DAYS = 30

print(f"\n📋 Configuration:")
print(f"   API Endpoint: {ESPN_TRANSACTIONS_URL}")
print(f"   Bronze Table: {BRONZE_TABLE}")
print(f"   Lookback Period: {LOOKBACK_DAYS} days")
print(f"   Request Timeout: {REQUEST_TIMEOUT}s")

print("\n" + "="*80)

In [0]:
# Install requests library
%pip install requests --quiet

print("✅ Dependencies installed")

In [0]:
# =============================================================================
# FETCH NFL TRANSACTIONS FROM ESPN API
# =============================================================================

import requests
import json
from datetime import datetime, timedelta
import hashlib
from typing import List, Dict, Optional

print("="*80)
print("📡 Fetching NFL Transactions")
print("="*80)

# Calculate date range
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"\n📅 Date Range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"\n⏳ Fetching transactions...\n")

# === FETCH FUNCTION ===
def fetch_nfl_transactions(timeout: int = REQUEST_TIMEOUT) -> Optional[Dict]:
    """Fetch NFL transactions from ESPN API."""
    try:
        response = requests.get(ESPN_TRANSACTIONS_URL, timeout=timeout)
        
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except Exception as e:
        print(f"❌ Error fetching transactions: {str(e)}")
        return None

# Fetch transactions
data = fetch_nfl_transactions()

if not data or 'items' not in data:
    print("❌ No transaction data returned from API")
    all_transactions = []
else:
    transactions = data.get('items', [])
    
    print(f"✅ Fetched {len(transactions)} total transactions from API")
    
    # === PARSE TRANSACTIONS ===
    all_transactions = []
    
    for txn in transactions:
        try:
            # Extract basic fields
            transaction_id = str(txn.get('id', ''))
            transaction_date_str = txn.get('date', '')
            transaction_type = txn.get('type', '')
            description = txn.get('description', '')
            
            # Parse date
            transaction_date = None
            if transaction_date_str:
                try:
                    transaction_date = datetime.fromisoformat(transaction_date_str.replace('Z', '+00:00'))
                except:
                    pass
            
            # Filter by date range
            if transaction_date and transaction_date < start_date:
                continue
            
            # Extract player info (if available)
            athletes = txn.get('athletes', [])
            
            if athletes:
                for athlete in athletes:
                    player_name = athlete.get('displayName', '')
                    espn_player_id = str(athlete.get('id', ''))
                    position = athlete.get('position', {}).get('abbreviation', '')
                    
                    # Extract team
                    team_code = ''
                    team_obj = athlete.get('team', {})
                    if team_obj:
                        team_code = team_obj.get('abbreviation', '')
                    
                    # Build transaction record
                    transaction_record = {
                        'transaction_id': transaction_id,
                        'transaction_date': transaction_date.isoformat() + 'Z' if transaction_date else '',
                        'transaction_type': transaction_type,
                        'player_name': player_name,
                        'position': position,
                        'team': team_code,
                        'description': description,
                        'espn_player_id': espn_player_id if espn_player_id != '0' else None,
                        'fetched_at': datetime.utcnow().isoformat() + 'Z'
                    }
                    
                    all_transactions.append(transaction_record)
            else:
                # Transaction without specific player (team-level)
                transaction_record = {
                    'transaction_id': transaction_id,
                    'transaction_date': transaction_date.isoformat() + 'Z' if transaction_date else '',
                    'transaction_type': transaction_type,
                    'player_name': None,
                    'position': None,
                    'team': None,
                    'description': description,
                    'espn_player_id': None,
                    'fetched_at': datetime.utcnow().isoformat() + 'Z'
                }
                
                all_transactions.append(transaction_record)
                
        except Exception as e:
            print(f"⚠️  Error parsing transaction: {str(e)}")
            continue
    
    print(f"\n📊 Parse Results:")
    print(f"   Transactions in Date Range: {len(all_transactions)}")
    
    # Show transaction type breakdown
    if all_transactions:
        from collections import Counter
        type_counts = Counter([t['transaction_type'] for t in all_transactions])
        print(f"\n📋 Transaction Types:")
        for txn_type, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"   {txn_type}: {count}")
    
    print(f"\n✅ Successfully parsed {len(all_transactions)} transactions")

In [0]:
%sql
-- Create bronze table for NFL transactions (if not exists)

CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_transactions (
  transaction_id STRING NOT NULL COMMENT 'Unique transaction ID',
  transaction_date TIMESTAMP COMMENT 'Date of transaction',
  transaction_type STRING COMMENT 'Type (Signed, Waived, IR, etc.)',
  player_name STRING COMMENT 'Player name',
  position STRING COMMENT 'Player position',
  team STRING COMMENT 'Team (from or to)',
  description STRING COMMENT 'Full transaction description',
  espn_player_id STRING COMMENT 'ESPN player ID (if available)',
  fetched_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp',
  CONSTRAINT pk_nfl_transactions PRIMARY KEY (transaction_id, player_name)
)
COMMENT 'Official NFL transactions from ESPN API - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

DESCRIBE EXTENDED main.fantasai.bronze_nfl_transactions;

In [0]:
# =============================================================================
# WRITE TO BRONZE TABLE WITH DEDUPLICATION
# =============================================================================

from pyspark.sql import functions as F

print("="*80)
print("💾 Writing Transactions to Bronze Table")
print("="*80)

if len(all_transactions) == 0:
    print("\n⚠️  No transactions to write. Skipping write operation.")
else:
    # Convert to Spark DataFrame
    transactions_df = spark.createDataFrame(all_transactions)
    
    # Convert timestamp strings to proper timestamps
    transactions_df = transactions_df \
        .withColumn('transaction_date', F.to_timestamp('transaction_date')) \
        .withColumn('fetched_at', F.to_timestamp('fetched_at'))
    
    print(f"\n📊 Prepared {transactions_df.count()} transactions for insertion")
    
    # Check for existing transactions (deduplication)
    existing_transactions_df = spark.sql(f"""
        SELECT DISTINCT transaction_id, 
               COALESCE(player_name, 'TEAM_TXN') as player_name
        FROM {BRONZE_TABLE}
    """)
    
    existing_count = existing_transactions_df.count()
    print(f"📋 Found {existing_count} existing transactions in database")
    
    # Handle null player_names for join
    transactions_df = transactions_df.withColumn(
        'player_name_key',
        F.coalesce(F.col('player_name'), F.lit('TEAM_TXN'))
    )
    
    # Left anti join to find new transactions only
    new_transactions_df = transactions_df.join(
        existing_transactions_df,
        (transactions_df.transaction_id == existing_transactions_df.transaction_id) &
        (transactions_df.player_name_key == existing_transactions_df.player_name),
        how='left_anti'
    ).drop('player_name_key')
    
    new_count = new_transactions_df.count()
    duplicate_count = transactions_df.count() - new_count
    
    print(f"\n🔍 Deduplication results:")
    print(f"   New transactions: {new_count}")
    print(f"   Duplicates skipped: {duplicate_count}")
    
    if new_count > 0:
        print(f"\n💾 Writing {new_count} new transactions to {BRONZE_TABLE}...")
        
        new_transactions_df.write \
            .mode('append') \
            .saveAsTable(BRONZE_TABLE)
        
        print("\n✅ Write complete!")
        
        # Show sample of new transactions
        print("\n💼 Sample of new transactions:")
        new_transactions_df.select(
            'transaction_date',
            'transaction_type',
            'player_name',
            'position',
            'team',
            F.substring('description', 1, 60).alias('description_preview')
        ).orderBy(F.desc('transaction_date')).show(10, truncate=False)
    else:
        print("\n✓ No new transactions to write (all duplicates)")

print("\n" + "="*80)

In [0]:
%sql
-- Show most recent NFL transactions

SELECT 
  transaction_date,
  transaction_type,
  player_name,
  position,
  team,
  SUBSTRING(description, 1, 100) as description_preview,
  DATEDIFF(DAY, transaction_date, CURRENT_TIMESTAMP()) as days_ago
FROM main.fantasai.bronze_nfl_transactions
WHERE transaction_date IS NOT NULL
ORDER BY transaction_date DESC
LIMIT 25;

In [0]:
%sql
-- Summary statistics for NFL transactions table

SELECT 
  COUNT(*) as total_transactions,
  COUNT(DISTINCT player_name) as unique_players,
  COUNT(DISTINCT transaction_type) as transaction_types,
  COUNT(DISTINCT team) as teams_involved,
  MIN(transaction_date) as oldest_transaction,
  MAX(transaction_date) as newest_transaction,
  MAX(fetched_at) as last_ingestion_run,
  SUM(CASE WHEN espn_player_id IS NOT NULL THEN 1 ELSE 0 END) as transactions_with_player_ids
FROM main.fantasai.bronze_nfl_transactions;